# IOI Circuit Replication (GPT-2 small)

Notebook-first workspace for replicating the Indirect Object Identification
circuit from Wang et al., 2022 ("Interpretability in the Wild").

Setup is verified by `../smoke_test.py`. The prompt set and tokenization
guards live in `../ioi_prompts.py` so the same vetted prompts are shared
between the smoke test and the experiments here.

**Metric:** `logit[IO] - logit[S]` at the final position. Positive = the model
prefers the indirect object (correct).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # import the repo-root modules

import torch
from transformer_lens import HookedTransformer
# from ioi_prompts import clean_prompts, verify_alignment
from ioi_prompts import ioi_prompts, verify_alignment

torch.set_grad_enabled(False)
model = HookedTransformer.from_pretrained("gpt2", device="cpu")
verify_alignment(model)  # fail loudly if names/positions are misaligned
# sum(1 for _ in clean_prompts())  # count of clean examples, computed on the fly
sum(1 for _ in ioi_prompts())  # count of clean prompts (one per IOIPrompt)

In [ ]:
# Clean logit difference per prompt
def logit_diff(tokens, io_id, s_id):
    final = model(tokens)[0, -1]
    return (final[io_id] - final[s_id]).item()

# for text, io, s in clean_prompts():
#     io_id, s_id = model.to_single_token(io), model.to_single_token(s)
#     d = logit_diff(model.to_tokens(text), io_id, s_id)
#     print(f"{d:+.3f}  {text!r}")
for p in ioi_prompts():
    io_id, s_id = model.to_single_token(p.io), model.to_single_token(p.s)
    d = logit_diff(model.to_tokens(p.clean), io_id, s_id)
    print(f"{d:+.3f}  {p.clean!r}")

In [ ]:
# Attention patterns for one prompt (circuitsvis). Look at late-layer heads
# (9-11) for name-mover behavior attending from the final token to the IO name.
import circuitsvis as cv
# text, io, s = next(clean_prompts())
text = next(ioi_prompts()).clean
tokens = model.to_tokens(text)
_, cache = model.run_with_cache(tokens)
str_tokens = model.to_str_tokens(text)
layer = 9
cv.attention.attention_patterns(tokens=str_tokens, attention=cache["pattern", layer][0])

## The experiment (your desk work)

Below is where the actual replication happens — a full activation-patching
sweep, not the single-site plumbing check in `smoke_test.py`. Sketch:

1. Cache clean and corrupted activations for each prompt pair.
2. For each `(layer, position)` (and later, each head), patch the clean
   activation into the corrupted run and record the recovered logit diff.
3. Normalize: `(patched - corrupted) / (clean - corrupted)` so 0 = no effect,
   1 = fully restored. Plot as a layer × position heatmap.

Decisions worth making deliberately: which hook point (`resid_pre` vs
`z`/`attn_out` for per-head), whether to average the metric over the prompt
set or analyze per-template, and how to handle the BOS position.

In [ ]:
# TODO(you): activation-patching sweep over (layer, position) -> heatmap.
